<a href="https://colab.research.google.com/github/Yashhhh793/SIH26001-Landslide-Early-Warning/blob/main/SIH_Landslide_Early_Warning_TEAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ==========================================
# SIH LANDSLIDE EARLY WARNING SYSTEM
# TEAM VERSION
# ==========================================

# MODULE 1: SETUP

import pandas as pd
import numpy as np
import requests
import math
import json
import os
import joblib

from sklearn.ensemble import GradientBoostingClassifier

print("✅ Libraries loaded successfully")
print("🚨 SIH Landslide Early Warning System")

✅ Libraries loaded successfully
🚨 SIH Landslide Early Warning System


STEP 2 — Configuration

In [6]:
# ==========================================
# MODULE 2: CONFIGURATION
# ==========================================

LOCATION_COORDS = {
    "Assam": (26.1445, 91.7362),
    "Arunachal Pradesh": (27.0844, 93.6053),
    "Manipur": (24.8170, 93.9368),
    "Meghalaya": (25.4670, 91.3662),
    "Mizoram": (23.1645, 92.9376),
    "Nagaland": (25.6751, 94.1086),
    "Sikkim": (27.5330, 88.5122),
    "Tripura": (23.9408, 91.9882)
}

ML_FEATURES = [
    "rainfall_3day_mm",
    "rainfall_7day_mm",
    "slope_degrees",
    "soil_moisture_m3_m3",
    "elevation_m"
]

print("✅ Configuration loaded")
print("📍 Locations:", len(LOCATION_COORDS))
print("🤖 ML Features:", ML_FEATURES)

✅ Configuration loaded
📍 Locations: 8
🤖 ML Features: ['rainfall_3day_mm', 'rainfall_7day_mm', 'slope_degrees', 'soil_moisture_m3_m3', 'elevation_m']


Nasa Historical data set

In [9]:
# ==========================================
# MODULE 3: NASA LANDSLIDE DATA
# ==========================================

NASA_URL = "https://data.nasa.gov/docs/legacy/Global_Landslide_Catalog_Export/Global_Landslide_Catalog_Export_rows.csv"

df_nasa = pd.read_csv(NASA_URL)

print("✅ NASA dataset loaded")
print("📊 Total records:", len(df_nasa))
print("📋 Total columns:", len(df_nasa.columns))

✅ NASA dataset loaded
📊 Total records: 11033
📋 Total columns: 31


Northeast India Filter

In [10]:
# ==========================================
# MODULE 4: NORTHEAST INDIA FILTER
# ==========================================

NE_STATES = [
    "Assam",
    "Arunachal Pradesh",
    "Manipur",
    "Meghalaya",
    "Mizoram",
    "Nagaland",
    "Sikkim",
    "Tripura"
]

ne_df = df_nasa[
    df_nasa["admin_division_name"].isin(NE_STATES)
].copy()

ne_df["event_date"] = pd.to_datetime(
    ne_df["event_date"],
    errors="coerce"
)

ne_df = ne_df.dropna(
    subset=["event_date", "latitude", "longitude"]
).reset_index(drop=True)

print("✅ Northeast India data filtered")
print("📊 Landslide events:", len(ne_df))
print("\nState-wise events:")
print(ne_df["admin_division_name"].value_counts())

✅ Northeast India data filtered
📊 Landslide events: 251

State-wise events:
admin_division_name
Assam                82
Manipur              56
Sikkim               31
Mizoram              27
Arunachal Pradesh    20
Meghalaya            18
Nagaland             14
Tripura               3
Name: count, dtype: int64


/tmp/ipykernel_4177/1080721284.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ne_df["event_date"] = pd.to_datetime(


Historical Modul Rainfall

In [ ]:
# ==========================================
# MODULE 5: HISTORICAL RAINFALL API
# ==========================================

def get_historical_rainfall(lat, lon, event_date):
    start_date = (
        pd.Timestamp(event_date) - pd.Timedelta(days=2)
    ).strftime("%Y-%m-%d")

    end_date = pd.Timestamp(event_date).strftime("%Y-%m-%d")

    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={start_date}"
        f"&end_date={end_date}"
        "&daily=precipitation_sum"
        "&timezone=auto"
    )

    response = requests.get(url, timeout=30)

    if response.status_code != 200:
        return None

    data = response.json()

    rainfall = data["daily"]["precipitation_sum"]

    return round(sum(rainfall), 2)


# Test with first NASA event
test_event = ne_df.iloc[0]

rainfall_test = get_historical_rainfall(
    test_event["latitude"],
    test_event["longitude"],
    test_event["event_date"]
)

print("📅 Event date:", test_event["event_date"].date())
print("🌧️ 3-Day rainfall:", rainfall_test, "mm")

Historicl Moisture API

In [ ]:
# ==========================================
# MODULE 6: HISTORICAL SOIL MOISTURE
# ==========================================

def get_historical_soil_moisture(lat, lon, event_date):
    """
    Fetches hourly soil moisture for the event date
    and calculates the daily average.
    """

    date_str = pd.Timestamp(event_date).strftime("%Y-%m-%d")

    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}"
        f"&longitude={lon}"
        f"&start_date={date_str}"
        f"&end_date={date_str}"
        "&hourly=soil_moisture_0_to_7cm"
        "&timezone=auto"
    )

    try:
        response = requests.get(url, timeout=30)

        if response.status_code != 200:
            print(
                f"⚠️ API error {response.status_code} "
                f"for {date_str}"
            )
            return np.nan

        data = response.json()

        moisture = data["hourly"]["soil_moisture_0_to_7cm"]

        valid_values = [
            value for value in moisture
            if value is not None
        ]

        if len(valid_values) == 0:
            return np.nan

        return round(float(np.mean(valid_values)), 4)

    except Exception as e:
        print(f"⚠️ Request failed: {e}")
        return np.nan


print("✅ Historical soil-moisture function ready")

ELEVATION and CORRECT SLOPE

In [ ]:
# ==========================================
# MODULE 7: TERRAIN, ELEVATION & SLOPE
# ==========================================

def get_terrain_features(lat, lon):
    """
    Fetch elevation at 5 surrounding points
    and calculate terrain slope.
    """

    points = [
        (lat, lon),                  # Center
        (lat + 0.01, lon),           # North
        (lat - 0.01, lon),           # South
        (lat, lon + 0.01),           # East
        (lat, lon - 0.01)            # West
    ]

    try:
        latitudes = ",".join(str(p[0]) for p in points)
        longitudes = ",".join(str(p[1]) for p in points)

        url = (
            "https://api.open-meteo.com/v1/elevation"
            f"?latitude={latitudes}"
            f"&longitude={longitudes}"
        )

        response = requests.get(url, timeout=30)

        if response.status_code != 200:
            print(f"⚠️ Elevation API error: {response.status_code}")
            return np.nan, np.nan

        data = response.json()

        elevations = data["elevation"]

        if len(elevations) != 5:
            print("⚠️ Incorrect elevation data received")
            return np.nan, np.nan

        center = elevations[0]
        north = elevations[1]
        south = elevations[2]
        east = elevations[3]
        west = elevations[4]

        # Distances between points in metres
        north_south_distance = 2224.0
        east_west_distance = 2015.0

        # Elevation gradients
        dz_dy = (north - south) / north_south_distance
        dz_dx = (east - west) / east_west_distance

        # Slope calculation
        slope = math.degrees(
            math.atan(
                math.sqrt(dz_dx**2 + dz_dy**2)
            )
        )

        return round(float(center), 2), round(float(slope), 2)

    except Exception as e:
        print(f"⚠️ Terrain request failed: {e}")
        return np.nan, np.nan


print("✅ Terrain and slope function ready")

POSITIVE LANDSLIDE SAMPLES

In [11]:
# ==========================================
# MODULE 8: POSITIVE LANDSLIDE SAMPLES
# ==========================================

positive_df = ne_df[
    [
        "admin_division_name",
        "event_date",
        "latitude",
        "longitude"
    ]
].copy()

# Rename state column
positive_df = positive_df.rename(
    columns={
        "admin_division_name": "state_clean"
    }
)

# Positive class = recorded landslide
positive_df["label"] = 1

print("✅ Positive samples prepared")
print("📊 Positive samples:", len(positive_df))
print("\nColumns:")
print(positive_df.columns.tolist())

print("\nLabel distribution:")
print(positive_df["label"].value_counts())

✅ Positive samples prepared
📊 Positive samples: 251

Columns:
['state_clean', 'event_date', 'latitude', 'longitude', 'label']

Label distribution:
label
1    251
Name: count, dtype: int64


CONTROL/ BACKGROUND SAMPLE

In [13]:
# ==========================================
# MODULE 9: CONTROL / BACKGROUND SAMPLES
# ==========================================

def has_nearby_landslide(lat, lon, event_date, radius_km=10):
    """
    Checks whether NASA catalog contains a recorded
    landslide within approximately radius_km on the date.
    """

    same_date = ne_df[
        ne_df["event_date"].dt.date == pd.Timestamp(event_date).date()
    ].copy()

    if same_date.empty:
        return False

    # Approximate distance calculation
    lat_diff = (same_date["latitude"] - lat) * 111
    lon_diff = (
        (same_date["longitude"] - lon)
        * 111
        * math.cos(math.radians(lat))
    )

    distance_km = np.sqrt(
        lat_diff**2 + lon_diff**2
    )

    return (distance_km <= radius_km).any()


candidate_offsets = [-90, -60, -30, 30, 60, 90]

control_rows = []

for _, event in positive_df.iterrows():

    selected_date = None

    for offset in candidate_offsets:

        candidate_date = (
            event["event_date"]
            + pd.Timedelta(days=offset)
        )

        if not has_nearby_landslide(
            event["latitude"],
            event["longitude"],
            candidate_date
        ):
            selected_date = candidate_date
            break

    if selected_date is not None:
        control_rows.append({
            "state_clean": event["state_clean"],
            "event_date": selected_date,
            "latitude": event["latitude"],
            "longitude": event["longitude"],
            "label": 0
        })


control_df = pd.DataFrame(control_rows)

print("✅ Control/background samples prepared")
print("📊 Control samples:", len(control_df))

print("\nLabel distribution:")
print(control_df["label"].value_counts())

✅ Control/background samples prepared
📊 Control samples: 251

Label distribution:
label
0    251
Name: count, dtype: int64


HISTORICAL API FEATURES

In [ ]:
# ==========================================
# MODULE 10: HISTORICAL API FEATURES
# ==========================================

def get_historical_features(lat, lon, event_date):
    """
    Collect historical rainfall and soil-moisture
    features for one location and date.
    """

    event_date = pd.Timestamp(event_date)

    start_date = (
        event_date - pd.Timedelta(days=6)
    ).strftime("%Y-%m-%d")

    end_date = event_date.strftime("%Y-%m-%d")

    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}"
        f"&longitude={lon}"
        f"&start_date={start_date}"
        f"&end_date={end_date}"
        "&daily=precipitation_sum"
        "&hourly=soil_moisture_0_to_7cm"
        "&timezone=auto"
    )

    try:
        response = requests.get(url, timeout=30)

        if response.status_code != 200:
            return {
                "rainfall_3day_mm": np.nan,
                "rainfall_7day_mm": np.nan,
                "soil_moisture_m3_m3": np.nan
            }

        data = response.json()

        # 7-day rainfall
        rainfall = data["daily"]["precipitation_sum"]
        rainfall = [
            x for x in rainfall if x is not None
        ]

        rainfall_7day = sum(rainfall)

        # Last 3 days = 3-day rainfall
        rainfall_3day = sum(rainfall[-3:])

        # Event-day average soil moisture
        moisture = data["hourly"]["soil_moisture_0_to_7cm"]
        moisture = [
            x for x in moisture if x is not None
        ]

        soil_moisture = (
            np.mean(moisture)
            if moisture else np.nan
        )

        return {
            "rainfall_3day_mm": round(
                float(rainfall_3day), 2
            ),
            "rainfall_7day_mm": round(
                float(rainfall_7day), 2
            ),
            "soil_moisture_m3_m3": round(
                float(soil_moisture), 4
            )
        }

    except Exception as e:

        print("⚠️ API request failed:", e)

        return {
            "rainfall_3day_mm": np.nan,
            "rainfall_7day_mm": np.nan,
            "soil_moisture_m3_m3": np.nan
        }


print("✅ Historical API feature function ready")

FINAL  ML  DATASET

In [ ]:
# ==========================================
# MODULE 11: FINAL ML DATASET
# ==========================================

# Positive and control samples ko combine karna
final_df = pd.concat(
    [positive_df, control_df],
    ignore_index=True
)

# ML features
ML_FEATURES = [
    "rainfall_3day_mm",
    "rainfall_7day_mm",
    "slope_degrees",
    "soil_moisture_m3_m3",
    "elevation_m"
]

print("📊 Final dataset structure created")
print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))

print("\nRequired ML features:")
for feature in ML_FEATURES:
    print("•", feature)

print("\nLabels:")
print(final_df["label"].value_counts())

MACHINE LEARNING

In [ ]:
# ==========================================
# MODULE 12: MACHINE LEARNING MODEL
# ==========================================

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Required ML features
ML_FEATURES = [
    "rainfall_3day_mm",
    "rainfall_7day_mm",
    "slope_degrees",
    "soil_moisture_m3_m3",
    "elevation_m"
]

# Input and target
X = final_df[ML_FEATURES]
y = final_df["label"]

# Group by location so samples from the same location
# do not unnecessarily appear in both train and test sets.
groups = (
    final_df["latitude"].round(2).astype(str)
    + "_"
    + final_df["longitude"].round(2).astype(str)
)

# Group-aware train/test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Gradient Boosting model
model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

# Train
model.fit(X_train, y_train)

print("✅ Gradient Boosting model trained")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

MODEL EVALUATION


In [ ]:
# ==========================================
# MODULE 13: MODEL EVALUATION
# ==========================================

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print("==========================================")
print("       MODEL PERFORMANCE")
print("==========================================")

print(f"Accuracy  : {accuracy:.2%}")
print(f"Precision : {precision:.2%}")
print(f"Recall    : {recall:.2%}")
print(f"F1 Score  : {f1:.2%}")
print(f"ROC-AUC   : {roc_auc:.3f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Background",
            "Landslide"
        ],
        zero_division=0
    )
)

SAVE TRAINED MODEL

In [ ]:
# ==========================================
# MODULE 14: SAVE TRAINED MODEL
# ==========================================

MODEL_FILE = "landslide_production_model.pkl"

joblib.dump(
    model,
    MODEL_FILE
)

print("✅ Model saved successfully")
print("📁 File:", MODEL_FILE)

LIVE WEATHER DATA

In [ ]:
# ==========================================
# MODULE 15: LIVE WEATHER DATA
# ==========================================

def get_live_weather(location):
    """
    Fetch recent rainfall and current soil moisture
    for the selected Northeast India location.
    """

    lat, lon = LOCATION_COORDS[location]

    try:
        # Recent historical rainfall
        archive_url = (
            "https://archive-api.open-meteo.com/v1/archive"
            f"?latitude={lat}"
            f"&longitude={lon}"
            "&start_date=2026-08-30"
            "&end_date=2026-09-05"
            "&daily=precipitation_sum"
            "&timezone=auto"
        )

        archive_response = requests.get(
            archive_url,
            timeout=30
        )

        if archive_response.status_code != 200:
            raise Exception(
                f"Rainfall API error: "
                f"{archive_response.status_code}"
            )

        archive_data = archive_response.json()

        rainfall = archive_data["daily"]["precipitation_sum"]

        rainfall = [
            x for x in rainfall
            if x is not None
        ]

        rainfall_7day = sum(rainfall)
        rainfall_3day = sum(rainfall[-3:])

        # Current soil moisture
        forecast_url = (
            "https://api.open-meteo.com/v1/forecast"
            f"?latitude={lat}"
            f"&longitude={lon}"
            "&current=soil_moisture_0_to_7cm"
            "&timezone=auto"
        )

        forecast_response = requests.get(
            forecast_url,
            timeout=30
        )

        if forecast_response.status_code != 200:
            raise Exception(
                f"Soil moisture API error: "
                f"{forecast_response.status_code}"
            )

        forecast_data = forecast_response.json()

        soil_moisture = forecast_data[
            "current"
        ]["soil_moisture_0_to_7cm"]

        return (
            round(rainfall_3day, 2),
            round(rainfall_7day, 2),
            round(float(soil_moisture), 4)
        )

    except Exception as e:

        print("⚠️ Live weather request failed:")
        print(e)

        return None, None, None


print("✅ Live weather function ready")

LIVE TERRAIN DATA

In [ ]:
# ==========================================
# MODULE 16: LIVE TERRAIN DATA
# ==========================================

def get_live_terrain(location):
    """
    Fetch live terrain elevation and calculate slope
    using 5 surrounding elevation points.
    """

    lat, lon = LOCATION_COORDS[location]

    points = [
        (lat, lon),                  # Center
        (lat + 0.01, lon),           # North
        (lat - 0.01, lon),           # South
        (lat, lon + 0.01),           # East
        (lat, lon - 0.01)            # West
    ]

    try:
        latitudes = ",".join(str(p[0]) for p in points)
        longitudes = ",".join(str(p[1]) for p in points)

        url = (
            "https://api.open-meteo.com/v1/elevation"
            f"?latitude={latitudes}"
            f"&longitude={longitudes}"
        )

        response = requests.get(url, timeout=30)

        if response.status_code != 200:
            raise Exception(
                f"Elevation API error: {response.status_code}"
            )

        data = response.json()
        elevations = data["elevation"]

        if len(elevations) != 5:
            raise Exception("Invalid elevation response")

        center = elevations[0]
        north = elevations[1]
        south = elevations[2]
        east = elevations[3]
        west = elevations[4]

        # Distances in metres
        dy = 2224.0
        dx = 2015.0

        # Elevation gradients
        dz_dy = (north - south) / dy
        dz_dx = (east - west) / dx

        # Correct slope calculation
        slope = math.degrees(
            math.atan(
                math.sqrt(dz_dx**2 + dz_dy**2)
            )
        )

        return (
            round(float(center), 2),
            round(float(slope), 2)
        )

    except Exception as e:

        print("⚠️ Live terrain request failed:")
        print(e)

        return None, None


print("✅ Live terrain function ready")

LIVE RISK PREDICTION

In [ ]:
# ==========================================
# MODULE 17: LIVE RISK PREDICTION
# ==========================================

def predict_live_risk(location):
    """
    Combines live weather and terrain features
    and generates an AI-based landslide risk score.
    """

    # Get live weather
    rainfall_3day, rainfall_7day, soil_moisture = (
        get_live_weather(location)
    )

    # Get live terrain
    elevation, slope = get_live_terrain(location)

    # Check API data
    values = [
        rainfall_3day,
        rainfall_7day,
        soil_moisture,
        elevation,
        slope
    ]

    if any(value is None for value in values):
        return {
            "status": "API_UNAVAILABLE",
            "risk_score": None,
            "rainfall_3day_mm": rainfall_3day,
            "rainfall_7day_mm": rainfall_7day,
            "soil_moisture_m3_m3": soil_moisture,
            "slope_degrees": slope,
            "elevation_m": elevation
        }

    # Create model input
    input_data = pd.DataFrame(
        [[
            rainfall_3day,
            rainfall_7day,
            slope,
            soil_moisture,
            elevation
        ]],
        columns=ML_FEATURES
    )

    # AI prediction
    risk_score = model.predict_proba(
        input_data
    )[0][1]

    prediction = model.predict(
        input_data
    )[0]

    return {
        "status": "SUCCESS",
        "risk_score": round(
            float(risk_score), 4
        ),
        "prediction": int(prediction),
        "rainfall_3day_mm": rainfall_3day,
        "rainfall_7day_mm": rainfall_7day,
        "soil_moisture_m3_m3": soil_moisture,
        "slope_degrees": slope,
        "elevation_m": elevation
    }


print("✅ Live risk prediction function ready")

RISK LEVEL CLASSIFICATION

In [ ]:
# ==========================================
# MODULE 18: RISK LEVEL CLASSIFICATION
# ==========================================

def get_risk_level(risk_score):
    """
    Converts AI risk score into a simple
    landslide risk category.
    """

    if risk_score is None:
        return "API UNAVAILABLE"

    risk_percentage = risk_score * 100

    if risk_percentage >= 70:
        return "HIGH RISK"

    elif risk_percentage >= 50:
        return "MODERATE RISK"

    else:
        return "LOWER RISK"


def get_risk_message(risk_score):
    """
    Generates a simple user-facing risk message.
    """

    level = get_risk_level(risk_score)

    if level == "HIGH RISK":
        return "🚨 HIGH LANDSLIDE RISK"

    elif level == "MODERATE RISK":
        return "⚠️ MODERATE LANDSLIDE RISK"

    elif level == "LOWER RISK":
        return "✅ LOWER LANDSLIDE RISK"

    else:
        return "⚠️ LIVE DATA UNAVAILABLE"


print("✅ Risk classification ready")

HISTORICAL LANDSLIDE EVENT

In [14]:
# ==========================================
# MODULE 19: HISTORICAL LANDSLIDE EVENTS
# ==========================================

history_df = ne_df[
    [
        "event_date",
        "admin_division_name",
        "landslide_trigger",
        "latitude",
        "longitude"
    ]
].copy()

history_df = history_df.rename(
    columns={
        "event_date": "Date",
        "admin_division_name": "State",
        "landslide_trigger": "Trigger",
        "latitude": "Latitude",
        "longitude": "Longitude"
    }
)

# Latest events first
history_df = history_df.sort_values(
    "Date",
    ascending=False
).reset_index(drop=True)

# Dashboard table: latest 10 events
history_table = history_df.head(10)

print("✅ Historical event table ready")
print("📊 Total recorded events:", len(history_df))

display(history_table)

✅ Historical event table ready
📊 Total recorded events: 251


,Date,State,Trigger,Latitude,Longitude
0,2016-10-15 00:00:00,Sikkim,rain,27.1695,88.3664
1,2016-10-15 00:00:00,Sikkim,downpour,27.1720,88.5292
2,2016-10-12 00:00:00,Sikkim,continuous_rain,27.2676,88.2891
3,2016-08-05 00:00:00,Arunachal Pradesh,downpour,27.5754,91.9754
4,2016-07-28 23:00:00,Mizoram,continuous_rain,23.5397,93.3783
5,2016-07-28 23:00:00,Nagaland,downpour,25.8702,94.7844
6,2016-07-28 00:00:00,Arunachal Pradesh,monsoon,27.5748,91.8644
7,2016-07-26 23:00:00,Nagaland,downpour,26.0101,94.5282
8,2016-07-25 12:30:00,Manipur,continuous_rain,25.4800,94.1383
9,2016-07-20 00:00:00,Assam,rain,26.1966,91.8020


Location MAP

In [15]:
# ==========================================
# MODULE 20.1: LOCATION MAP
# ==========================================

def create_map(location):
    """
    Creates an interactive map for the
    selected Northeast India location.
    """

    lat, lon = LOCATION_COORDS[location]

    m = folium.Map(
        location=[lat, lon],
        zoom_start=7
    )

    folium.Marker(
        [lat, lon],
        tooltip=location,
        popup=f"Landslide Monitoring: {location}"
    ).add_to(m)

    return m._repr_html_()


print("✅ Location map function ready")

✅ Location map function ready


DASHBOARD DATA

In [ ]:
# ==========================================
# MODULE 20.2: DASHBOARD DATA
# ==========================================

def get_dashboard_data(location):
    """
    Collects live weather, terrain and AI risk
    data for the selected location.
    """

    result = predict_live_risk(location)

    if result["status"] != "SUCCESS":
        return {
            "status": "API_UNAVAILABLE",
            "rainfall_3day": None,
            "rainfall_7day": None,
            "soil_moisture": None,
            "slope": None,
            "elevation": None,
            "risk_score": None,
            "risk_level": "API UNAVAILABLE"
        }

    risk_score = result["risk_score"]
    risk_level = get_risk_level(risk_score)

    return {
        "status": "SUCCESS",
        "rainfall_3day": result["rainfall_3day_mm"],
        "rainfall_7day": result["rainfall_7day_mm"],
        "soil_moisture": result["soil_moisture_m3_m3"],
        "slope": result["slope_degrees"],
        "elevation": result["elevation_m"],
        "risk_score": risk_score,
        "risk_level": risk_level
    }


print("✅ Dashboard data function ready")

RISK CARD

In [16]:
# ==========================================
# MODULE 20.3: RISK CARD
# ==========================================

def create_risk_card(risk_score, risk_level):

    if risk_score is None:
        return """
        <div style="padding:20px;border-radius:15px;
                    background:#f5f5f5;text-align:center;">
            <h2>⚠️ LIVE DATA UNAVAILABLE</h2>
            <p>Weather API is temporarily unavailable.</p>
        </div>
        """

    percentage = risk_score * 100

    if risk_level == "HIGH RISK":
        icon = "🚨"
        message = "High probability of landslide based on current environmental conditions."

    elif risk_level == "MODERATE RISK":
        icon = "⚠️"
        message = "Moderate landslide risk. Stay alert and follow local advisories."

    else:
        icon = "✅"
        message = "Lower landslide risk under current environmental conditions."

    return f"""
    <div style="
        padding:25px;
        border-radius:18px;
        border:1px solid #ddd;
        background:white;
        text-align:center;
        box-shadow:0 4px 15px rgba(0,0,0,0.08);
    ">
        <h2>🤖 AI Landslide Risk Assessment</h2>
        <h1>{icon} {risk_level}</h1>
        <h2>Risk Score: {percentage:.2f}%</h2>
        <p>{message}</p>
    </div>
    """


print("✅ Risk card ready")

✅ Risk card ready


DASHBOARD LAYOUT

In [ ]:
# ==========================================
# MODULE 20.4: DASHBOARD LAYOUT
# ==========================================

with gr.Blocks(
    title="Landslide Early Warning System"
) as dashboard_app:

    gr.Markdown("""
    # 🌋 Landslide Early Warning System
    ### AI-Based Risk Monitoring for Northeast India

    **SIH26001 | Science • AI • Safer Communities**
    """)

    with gr.Row():

        location = gr.Dropdown(
            choices=list(LOCATION_COORDS.keys()),
            value="Sikkim",
            label="📍 Select Location"
        )

        refresh_btn = gr.Button(
            "🔄 Refresh Risk Data"
        )

    with gr.Row():

        rain3 = gr.Textbox(
            label="🌧️ 3-Day Rainfall"
        )

        rain7 = gr.Textbox(
            label="🌧️ 7-Day Rainfall"
        )

        soil = gr.Textbox(
            label="💧 Soil Moisture"
        )

    with gr.Row():

        slope = gr.Textbox(
            label="📐 Slope"
        )

        elevation = gr.Textbox(
            label="⛰️ Elevation"
        )

    risk_card = gr.HTML(
        value=create_risk_card(None, "API UNAVAILABLE")
    )

    gr.Markdown("## 🗺️ Monitoring Location")

    map_output = gr.HTML(
        value=create_map("Sikkim")
    )

    gr.Markdown("## 📋 Historical Landslide Events")

    history_output = gr.Dataframe(
        value=history_table,
        interactive=False
    )

    gr.Markdown("""
    ---
    **Landslide Early Warning System | SIH26001**

    NASA Landslide Data • Weather Data • Terrain Analysis
    """)

print("✅ Dashboard layout created")

DASHBOARD CALLBACK

In [ ]:
# ==========================================
# MODULE 20.5: DASHBOARD CALLBACK
# ==========================================

def update_dashboard(location):

    data = get_dashboard_data(location)

    if data["status"] != "SUCCESS":

        return (
            "Unavailable",
            "Unavailable",
            "Unavailable",
            "Unavailable",
            "Unavailable",
            create_risk_card(
                None,
                "API UNAVAILABLE"
            ),
            create_map(location),
            history_table
        )

    return (
        f"{data['rainfall_3day']:.2f} mm",
        f"{data['rainfall_7day']:.2f} mm",
        f"{data['soil_moisture']:.4f}",
        f"{data['slope']:.2f}°",
        f"{data['elevation']:.2f} m",
        create_risk_card(
            data["risk_score"],
            data["risk_level"]
        ),
        create_map(location),
        history_table
    )


dashboard_outputs = [
    rain3,
    rain7,
    soil,
    slope,
    elevation,
    risk_card,
    map_output,
    history_output
]

refresh_btn.click(
    fn=update_dashboard,
    inputs=location,
    outputs=dashboard_outputs
)

location.change(
    fn=update_dashboard,
    inputs=location,
    outputs=dashboard_outputs
)

print("✅ Dashboard callbacks connected")

LAUNCH DASHBOARD

In [ ]:
# ==========================================
# MODULE 20.6: LAUNCH DASHBOARD
# ==========================================

dashboard_app.launch(
    share=True,
    debug=False
)

GOOGLE DRIVE BACKUP

In [17]:
# ==========================================
# MODULE 21: GOOGLE DRIVE BACKUP
# ==========================================

from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount("/content/drive")

# Project backup folder
BACKUP_DIR = (
    "/content/drive/MyDrive/"
    "Landslide_Early_Warning_Project"
)

os.makedirs(BACKUP_DIR, exist_ok=True)

# Files to backup
FILES_TO_BACKUP = [
    "landslide_final_dataset_corrected.csv",
    "historical_5point_terrain.csv",
    "landslide_production_model.pkl"
]

for file_name in FILES_TO_BACKUP:

    source = f"/content/{file_name}"
    destination = f"{BACKUP_DIR}/{file_name}"

    if os.path.exists(source):
        shutil.copy2(source, destination)
        print(f"✅ Backed up: {file_name}")
    else:
        print(f"⚠️ Not found yet: {file_name}")

print("\n📁 Backup folder:")
print(BACKUP_DIR)

Mounted at /content/drive
⚠️ Not found yet: landslide_final_dataset_corrected.csv
⚠️ Not found yet: historical_5point_terrain.csv
⚠️ Not found yet: landslide_production_model.pkl

📁 Backup folder:
/content/drive/MyDrive/Landslide_Early_Warning_Project
